# Download files (IVDRS & SUDORS)

Downloads every **direct-link** file on the workbook's Files tab (or a CSV saved from it) and saves it as
`FILES_DIR / file_name`, in the right folder with the right name.

It handles direct PDF links (including links that start a download instead of opening a page),
**Google Drive** share links, the old **luriechildrens.org** PDFs, and **articles** (it finds the PDF from the
article page, and tries the DOI link if the first link fails). If a site blocks plain downloading it retries
in a hidden browser.

**Not handled here:** Infogram, Looker Studio and Canva rows. Those are web pages; use
`Capture_Web_Reports.ipynb`. Anything this notebook can't get is listed as **manual** in the log, with the reason.

Keep `download_files.py` in the same folder as this notebook.

| Cell | What it does | When |
|---|---|---|
| 1 | Install packages + the browser | Once (same as the capture notebook) |
| 2 | Settings | Every session; edit first |
| 3 | Helper | Every session |
| 4 | **Test** on 4 records (one of each link type) | First |
| 5 | **Download everything** on the Files tab | After the test looks right |
| 6 | Review the log: what still needs doing by hand | After cell 5 |
| 7 | Optional: download the Not Open Access articles for review | Any time |

In [ ]:
%run organize_folders.py

In [1]:
%run organize_folders.py --move

  SKIP  output\staging_deposit_log.csv  ->  output\prism\staging_deposit_log.csv  (already exists there; left as is)
  moved output\staging_payloads.json  ->  output\prism\staging_payloads.json
  moved output\staging_payloads_dryrun.json  ->  output\prism\staging_payloads_dryrun.json
  moved output\staging_registry.json  ->  output\prism\staging_registry.json
  moved output\staging_status.csv  ->  output\prism\staging_status.csv


PermissionError: [WinError 32] The process cannot access the file because it is being used by another process: 'C:\\Users\\keg827\\Documents\\Github_Repos\\Galter_Prism_IVDRS_SUDORS_Post_Request\\output\\production_deposit_log.csv'

## Cell 1: Install (run once)

In [ ]:
import subprocess, sys
subprocess.run([sys.executable, "-m", "pip", "install", "requests", "playwright", "openpyxl", "pillow", "pypdf"], check=True)
subprocess.run([sys.executable, "-m", "playwright", "install", "chromium"], check=True)
print("Done. If this is the first install, restart the kernel before continuing.")

## Cell 2: Settings ✏️ edit these

Use either the workbook or a CSV. A CSV needs at least `record_id`, `file_name` and `Download from`
(`DOI link` and `Link type` are used if present). Leave `SOURCE_CSV = None` to read the workbook.

In [ ]:
from pathlib import Path

SCRIPT     = Path("download_files.py")
WORKBOOK   = Path("2026_10-22 IVDRS_SUDORS_Reconciled_Deposit_List.xlsx")
SOURCE_CSV = None            # e.g. Path("files_to_download.csv") to use a CSV instead of the workbook
FILES_DIR  = Path("files")   # files are saved to FILES_DIR / file_name
LOG        = Path("download_log.csv")

OVERWRITE   = False          # re-download files that already exist
USE_BROWSER = True           # retry blocked downloads in a hidden browser (slower, gets more files)

# Test records: Google Drive, direct PDF, old Lurie PDF, article found from its DOI only
TEST_IDS = ["IVSU-0030", "IVSU-0011", "IVSU-0032", "IVSU-0085"]

for p in (SCRIPT, SOURCE_CSV or WORKBOOK):
    print(f"{'found  ' if Path(p).exists() else 'MISSING'}  {Path(p).resolve()}")

## Cell 3: Helper (run every session)

In [ ]:
import subprocess, sys

def run_download(*extra, sheet="Files", log=None):
    src = ["--csv", SOURCE_CSV] if SOURCE_CSV else ["--workbook", WORKBOOK, "--sheet", sheet]
    cmd = [sys.executable, str(SCRIPT), *map(str, src), "--files-dir", str(FILES_DIR), "--log", str(log or LOG), *map(str, extra)]
    if OVERWRITE:       cmd.append("--overwrite")
    if not USE_BROWSER: cmd.append("--no-browser")
    proc = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    for line in proc.stdout:
        print(line, end="")
    proc.wait()
    print(f"\n[finished, exit code {proc.returncode}]")

print("Helper ready.")

## Cell 4: Test on 4 records

Open the 4 files it saves and check they are the right documents. (Run again safely: existing files are skipped.)

In [ ]:
run_download("--only", *TEST_IDS, log="download_log_test.csv")

## Cell 5: Download everything on the Files tab

About 100 files; it pauses a second between files, so expect several minutes. Safe to re-run after a
failure: files that already exist are skipped.

In [ ]:
run_download()

## Cell 6: Review the log

- **ok**: saved.
- **ok-check**: taken from the only PDF link on a web page; open it to confirm it's the right document.
- **manual**: get it by hand (the reason is shown), save it with the listed `file_name`.
- **skipped (exists)**: already in the folder.

In [ ]:
import csv
from collections import Counter

rows = list(csv.DictReader(open(LOG, encoding="utf-8")))
print(Counter(r["status"] for r in rows), "\n")
for status in ("manual", "ok-check"):
    sel = [r for r in rows if r["status"] == status]
    if sel:
        print(f"--- {status} ({len(sel)}) ---")
        for r in sel:
            print(f"{r['record_id']}  {r['file_name']}\n      {r['note'] or r['method']}\n      {r['url_used']}")
        print()

## Cell 7: Optional: Not Open Access articles (for review only, not for upload)

Saves to `FILES_DIR / not_open_access/`. Closed-access articles usually end up as **manual**: get those
through library access if you want them.

In [ ]:
run_download(sheet="Not Open Access Files", log="download_log_not_open_access.csv")